Processing categorical data

1. Load the dataset that resulted from the Cleaning data exercise.
2. Identify the categorical variables in the loaded data frame.
3. For the categorical variables, apply the conversion of text values to numeric values using the method of your choice.
4. Save the dataset to a .csv file. When saving, omit the index column (the index argument in the method that allows you to save the file to .csv).

In [1]:
import pandas as pd

heart_cleaned_df = pd.read_csv("data/heart_data_cleaned.csv")
heart_cleaned_df.head()

,age,restbp,chol,restecg,maxhr,oldpeak,slope,ca,thal,sex,chestpain,fbs,exang,ahd
0,63.0,145.0,233.0,2.0,150.0,2.3,3.0,0.0,fixed,Male,typical,Yes,No,No
1,67.0,160.0,286.0,2.0,108.0,1.5,2.0,3.0,normal,Male,asymptomatic,No,Yes,Yes
2,67.0,120.0,229.0,2.0,129.0,2.6,2.0,2.0,reversable,Male,asymptomatic,No,Yes,Yes
3,37.0,130.0,240.0,0.0,187.0,3.5,3.0,0.0,REversable,Male,nonanginal,No,No,No
4,41.0,130.0,204.0,2.0,172.0,1.4,1.0,0.0,FIXED,Female,nontypical,No,No,No


In [2]:
numeric_data = ['age', 'restbp', 'chol', 'maxhr', 'oldpeak', 'slope', 'ca']
heart_cleaned_numerical_df = heart_cleaned_df[numeric_data]
heart_cleaned_numerical_df.head()

,age,restbp,chol,maxhr,oldpeak,slope,ca
0,63.0,145.0,233.0,150.0,2.3,3.0,0.0
1,67.0,160.0,286.0,108.0,1.5,2.0,3.0
2,67.0,120.0,229.0,129.0,2.6,2.0,2.0
3,37.0,130.0,240.0,187.0,3.5,3.0,0.0
4,41.0,130.0,204.0,172.0,1.4,1.0,0.0


In [3]:
heart_cleaned_df["restecg"].unique()

array([2., 0., 1.])

In [4]:
categorical_data = ["thal", "sex", "chestpain", "fbs", "exang", "restecg","ahd"]
heart_cleaned_categorical = heart_cleaned_df[categorical_data]
heart_cleaned_categorical

,thal,sex,chestpain,fbs,exang,restecg,ahd
0,fixed,Male,typical,Yes,No,2.0,No
1,normal,Male,asymptomatic,No,Yes,2.0,Yes
2,reversable,Male,asymptomatic,No,Yes,2.0,Yes
3,REversable,Male,nonanginal,No,No,0.0,No
4,FIXED,Female,nontypical,No,No,2.0,No
...,...,...,...,...,...,...,...
296,reversable,Male,typical,No,No,0.0,Yes
297,reversable,Male,asymptomatic,Yes,No,0.0,Yes
298,reversable,Male,asymptomatic,No,Yes,0.0,Yes
299,normal,Female,nontypical,No,No,2.0,Yes


In [5]:
heart_cleaned_categorical.nunique()

thal         8
sex          2
chestpain    7
fbs          2
exang        2
restecg      3
ahd          3
dtype: int64

In [6]:
# Normalize every text/object-like value in the categorical dataframe to lowercase
heart_cleaned_categorical = heart_cleaned_categorical.apply(lambda col: col.map(lambda x: x.lower() if isinstance(x, str) else x))
heart_cleaned_categorical.head()

,thal,sex,chestpain,fbs,exang,restecg,ahd
0,fixed,male,typical,yes,no,2.0,no
1,normal,male,asymptomatic,no,yes,2.0,yes
2,reversable,male,asymptomatic,no,yes,2.0,yes
3,reversable,male,nonanginal,no,no,0.0,no
4,fixed,female,nontypical,no,no,2.0,no


In [7]:
heart_cleaned_categorical["chestpain"]

0           typical
1      asymptomatic
2      asymptomatic
3        nonanginal
4        nontypical
           ...     
296         typical
297    asymptomatic
298    asymptomatic
299      nontypical
300      nonanginal
Name: chestpain, Length: 301, dtype: object

works in two layers:

apply(...)
It runs a function separately on every column in the DataFrame.

lambda col: ...
That function receives one column at a time.

col.map(...)
It goes through every value inside that column.

lambda x: x.lower() if isinstance(x, str) else x
For each value:

if the value is a string, convert it to lowercase
otherwise, leave it unchanged

In [8]:
# Encoding categorical data
# First normalize all text values to lowercase and trim spaces
heart_cleaned_categorical = heart_cleaned_categorical.apply(
    lambda col: col.map(lambda x: x.lower().strip() if isinstance(x, str) else x)
)

# Fix known spelling inconsistencies before pd.get_dummies()
heart_cleaned_categorical["chestpain"] = heart_cleaned_categorical["chestpain"].replace({
    "nontypica": "nontypical",
    "nontypiccal": "nontypical",
    "asymptomaticc": "asymptomatic"
})

heart_cleaned_categorical["thal"] = heart_cleaned_categorical["thal"].replace({
    "reversabble": "reversable"
})

# Encode categorical columns to numeric dummy variables
heart_cleaned_categorical = pd.get_dummies(
    data=heart_cleaned_categorical,
    columns=heart_cleaned_categorical.columns,
    drop_first=True,
    dtype=int
)
heart_cleaned_categorical

,thal_normal,thal_reversable,sex_male,chestpain_nonanginal,chestpain_nontypical,chestpain_typical,fbs_yes,exang_yes,restecg_1.0,restecg_2.0,ahd_yes
0,0,0,1,0,0,1,1,0,0,1,0
1,1,0,1,0,0,0,0,1,0,1,1
2,0,1,1,0,0,0,0,1,0,1,1
3,0,1,1,1,0,0,0,0,0,0,0
4,0,0,0,0,1,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...
296,0,1,1,0,0,1,0,0,0,0,1
297,0,1,1,0,0,0,1,0,0,0,1
298,0,1,1,0,0,0,0,1,0,0,1
299,1,0,0,0,1,0,0,0,0,1,1


In [9]:
heart_encoded_df = pd.concat([heart_cleaned_numerical_df, heart_cleaned_categorical], axis=True)
heart_encoded_df.head()

,age,restbp,chol,maxhr,oldpeak,slope,ca,thal_normal,thal_reversable,sex_male,chestpain_nonanginal,chestpain_nontypical,chestpain_typical,fbs_yes,exang_yes,restecg_1.0,restecg_2.0,ahd_yes
0,63.0,145.0,233.0,150.0,2.3,3.0,0.0,0,0,1,0,0,1,1,0,0,1,0
1,67.0,160.0,286.0,108.0,1.5,2.0,3.0,1,0,1,0,0,0,0,1,0,1,1
2,67.0,120.0,229.0,129.0,2.6,2.0,2.0,0,1,1,0,0,0,0,1,0,1,1
3,37.0,130.0,240.0,187.0,3.5,3.0,0.0,0,1,1,1,0,0,0,0,0,0,0
4,41.0,130.0,204.0,172.0,1.4,1.0,0.0,0,0,0,0,1,0,0,0,0,1,0


In [10]:
# Save the cleaned dataset to a CSV file without the index column
heart_encoded_df.to_csv("data/heart_data_encoded.csv", index=False)
print("Cleaned dataset saved to data/heart_data_encoded.csv")

Cleaned dataset saved to data/heart_data_encoded.csv
